# Задача 7. Детекция машинной генерации текста

Automated Text Identification: Multilingual Transformer-based Models Approach,
на данных SemEval 2024 Task 8 / Subtask A

- encoder-модели: XLM-RoBERTa, mDeBERTa V3, MiniLM
- classification head: 3 слоя + GELU + dropout
- cleaning
- expansion: длинные тексты от 450 символов режутся на куски не короче 50
- схема обучения на 5 эпохах
  1  head-only -> 3 full fine-tuning -> 1 head-only
- сравнение default и hrabovii


In [1]:
import copy
import html
import random
import re
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
import sentencepiece
from IPython.display import display
from sklearn.metrics import accuracy_score, classification_report, f1_score
from torch import nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import MultiStepLR
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from transformers import (
    AutoModel,
    AutoTokenizer,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
    set_seed,
)
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE


device(type='cuda')

In [2]:
cfg = SimpleNamespace(
    train_path="SemEval2024-task8/subtaskA/data/subtaskA_train_multilingual.jsonl",
    dev_path="SemEval2024-task8/subtaskA/data/subtaskA_dev_multilingual.jsonl",
    max_length=256,
    train_limit=12000,
    dev_limit=3000,
    batch_size=16,
    num_epochs=5,
    learning_rate=2e-5,
    encoder_learning_rate=1e-5,
    mdeberta_full_learning_rate=5e-6,
    mdeberta_encoder_learning_rate=5e-7,
    mdeberta_tunable_layers=2,
    weight_decay=0.01,
    dropout=0.2,
    warmup_ratio=0.1,
    grad_clip_norm=0.5,
    long_text_threshold=450,
    min_chunk_length=50,
    log_root="runs/task7/article_style",
    checkpoint_root="checkpoints/task7/article_style",
)

for path in [cfg.train_path, cfg.dev_path]:
    if not Path(path).exists():
        raise FileNotFoundError(f"Не найден файл: {path}")

raw_train_df = pd.read_json(cfg.train_path, lines=True)
raw_dev_df = pd.read_json(cfg.dev_path, lines=True)

def stratified_sample(frame: pd.DataFrame, limit: int) -> pd.DataFrame:
    if limit is None or limit >= len(frame):
        return frame.reset_index(drop=True)
    label_count = frame["label"].nunique()
    per_label = max(1, limit // label_count)
    parts = []
    for _, part in frame.groupby("label"):
        parts.append(part.sample(min(len(part), per_label), random_state=SEED))
    sampled = pd.concat(parts, ignore_index=True)
    return sampled.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

if cfg.train_limit:
    raw_train_df = stratified_sample(raw_train_df, cfg.train_limit)

if cfg.dev_limit:
    raw_dev_df = stratified_sample(raw_dev_df, cfg.dev_limit)

print("Raw train:", raw_train_df.shape)
print(raw_train_df["label"].value_counts().sort_index())
print("Raw dev:", raw_dev_df.shape)
print(raw_dev_df["label"].value_counts().sort_index())
raw_train_df.head(2)


Raw train: (12000, 5)
label
0    6000
1    6000
Name: count, dtype: int64
Raw dev: (3000, 5)
label
0    1500
1    1500
Name: count, dtype: int64


,text,label,model,source,id
0,密码保护问题答案可以重置，请按以下步骤来完成： \r\r一、到达新浪通行证首页后，输入您的登...,0,human,chinese,141440
1,Ленин води след спорен президентски вот в Еква...,1,chatGPT,bulgarian,73882


In [3]:
def clean_text(text: str) -> str:
    text = html.unescape(str(text))
    text = re.sub(r"@[A-Za-z0-9_]+", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def split_long_text(text: str, min_length: int = 450, min_chunk_length: int = 50) -> list[str]:
    if len(text) < min_length:
        return [text]

    sentences = re.split(r"(?<=[.!?])\s+", text)
    chunks = []
    current = ""

    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue
        candidate = (current + " " + sentence).strip()
        if len(candidate) < min_chunk_length:
            current = candidate
            continue
        if len(current) >= min_chunk_length:
            chunks.append(current)
            current = sentence
        else:
            current = candidate

    if current:
        if len(current) < min_chunk_length and chunks:
            chunks[-1] = (chunks[-1] + " " + current).strip()
        else:
            chunks.append(current)

    return chunks or [text]


def preprocess_frame(frame: pd.DataFrame, expand_long_texts: bool) -> pd.DataFrame:
    rows = []
    for row in frame.itertuples(index=False):
        cleaned_text = clean_text(row.text)
        pieces = split_long_text(
            cleaned_text,
            min_length=cfg.long_text_threshold,
            min_chunk_length=cfg.min_chunk_length,
        ) if expand_long_texts else [cleaned_text]
        for part_index, piece in enumerate(pieces):
            rows.append(
                {
                    "id": f"{row.id}_{part_index}" if expand_long_texts else row.id,
                    "label": int(row.label),
                    "text": piece,
                    "source": getattr(row, "source", ""),
                    "model": getattr(row, "model", ""),
                }
            )
    return pd.DataFrame(rows)


train_original_df = raw_train_df[["id", "label", "text", "source", "model"]].copy()
train_processed_df = preprocess_frame(raw_train_df, expand_long_texts=True)
dev_eval_df = preprocess_frame(raw_dev_df, expand_long_texts=False)

print("Original train size:", len(train_original_df))
print("Processed train size:", len(train_processed_df))
print("Eval size:", len(dev_eval_df))
display(train_processed_df.head(3))


Original train size: 12000
Processed train size: 204274
Eval size: 3000


,id,label,text,source,model
0,141440_0,0,密码保护问题答案可以重置，请按以下步骤来完成： 一、到达新浪通行证首页后，输入您的登录名及登...,chinese,human
1,73882_0,1,Ленин води след спорен президентски вот в Еква...,bulgarian,chatGPT
2,73882_1,1,"След дълги дни на протести и оспарявания, уста...",bulgarian,chatGPT


In [4]:
LABEL_NAMES = {0: "human", 1: "machine"}


def mean_pool(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    mask = attention_mask.unsqueeze(-1).type_as(last_hidden_state)
    masked_hidden = last_hidden_state * mask
    token_count = mask.sum(dim=1).clamp(min=1e-6)
    return masked_hidden.sum(dim=1) / token_count


class EncoderClassifier(nn.Module):
    def __init__(self, model_name: str, dropout: float = 0.2) -> None:
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        mid_size = max(hidden_size // 2, 128)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, mid_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mid_size, 2),
        )

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        encoder_kwargs = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
        }
        if token_type_ids is not None:
            encoder_kwargs["token_type_ids"] = token_type_ids
        outputs = self.encoder(**encoder_kwargs)
        pooled_embedding = mean_pool(outputs.last_hidden_state, attention_mask)
        pooled_embedding = pooled_embedding.to(self.classifier[0].weight.dtype)
        return self.classifier(pooled_embedding)


def set_encoder_trainability(model: EncoderClassifier, model_name: str, phase: str) -> None:
    for parameter in model.encoder.parameters():
        parameter.requires_grad = False

    if phase == "head":
        return

    if "mdeberta" in model_name.lower():
        for layer in model.encoder.encoder.layer[-cfg.mdeberta_tunable_layers:]:
            for parameter in layer.parameters():
                parameter.requires_grad = True
        return

    for parameter in model.encoder.parameters():
        parameter.requires_grad = True


def tokenize_frame(frame: pd.DataFrame, tokenizer, max_length: int) -> list[dict]:
    records = []
    for row in frame.itertuples(index=False):
        encoded = tokenizer(row.text, truncation=True, max_length=max_length)
        encoded["labels"] = int(row.label)
        records.append(encoded)
    return records


def make_loader(frame: pd.DataFrame, tokenizer, batch_size: int, shuffle: bool) -> DataLoader:
    records = tokenize_frame(frame, tokenizer, max_length=cfg.max_length)
    collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")
    return DataLoader(records, batch_size=batch_size, shuffle=shuffle, collate_fn=collator)


def epoch_phase(epoch_index: int) -> str:
    schedule = ["head", "full", "full", "full", "head"]
    return schedule[min(epoch_index, len(schedule) - 1)]


def resolve_phase(model_name: str, epoch_index: int) -> str:
    phase = epoch_phase(epoch_index)
    if "mdeberta" in model_name.lower():
        return "head"
    return phase


def phase_block_length(model_name: str, epoch_index: int) -> int:
    schedule = [resolve_phase(model_name, idx) for idx in range(cfg.num_epochs)]
    phase = schedule[epoch_index]
    block_length = 0
    for next_phase in schedule[epoch_index:]:
        if next_phase != phase:
            break
        block_length += 1
    return block_length


def split_weight_decay_parameters(module: nn.Module):
    no_decay_terms = ("bias", "LayerNorm.weight", "LayerNorm.bias", "layer_norm.weight", "layer_norm.bias")
    decay_params = []
    no_decay_params = []

    for name, parameter in module.named_parameters():
        if not parameter.requires_grad:
            continue
        if any(term in name for term in no_decay_terms):
            no_decay_params.append(parameter)
        else:
            decay_params.append(parameter)

    return decay_params, no_decay_params


def resolve_learning_rates(model_name: str, phase: str) -> tuple[float, float]:
    classifier_lr = cfg.learning_rate
    encoder_lr = cfg.encoder_learning_rate

    if "mdeberta" in model_name.lower() and phase == "full":
        classifier_lr = min(classifier_lr, cfg.mdeberta_full_learning_rate)
        encoder_lr = min(encoder_lr, cfg.mdeberta_encoder_learning_rate)

    return classifier_lr, encoder_lr


def build_optimizer(model: EncoderClassifier, model_name: str, phase: str):
    parameter_groups = []
    classifier_lr, encoder_lr = resolve_learning_rates(model_name, phase)

    classifier_decay, classifier_no_decay = split_weight_decay_parameters(model.classifier)
    if classifier_decay:
        parameter_groups.append(
            {"params": classifier_decay, "lr": classifier_lr, "weight_decay": cfg.weight_decay}
        )
    if classifier_no_decay:
        parameter_groups.append(
            {"params": classifier_no_decay, "lr": classifier_lr, "weight_decay": 0.0}
        )

    if phase == "full":
        encoder_decay, encoder_no_decay = split_weight_decay_parameters(model.encoder)
        if encoder_decay:
            parameter_groups.append(
                {"params": encoder_decay, "lr": encoder_lr, "weight_decay": cfg.weight_decay}
            )
        if encoder_no_decay:
            parameter_groups.append(
                {"params": encoder_no_decay, "lr": encoder_lr, "weight_decay": 0.0}
            )

    return AdamW(parameter_groups)


def build_scheduler(optimizer, total_steps: int):
    warmup_steps = max(1, int(total_steps * cfg.warmup_ratio))
    return get_linear_schedule_with_warmup(
        optimizer=optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )


def compute_metrics(labels: list[int], preds: list[int]) -> dict:
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds),
    }


def validate_finite_tensor(tensor: torch.Tensor, message: str) -> None:
    if not torch.isfinite(tensor).all():
        raise RuntimeError(message)


def evaluate_model(model, loader, loss_fn):
    model.eval()
    losses = []
    all_labels = []
    all_preds = []
    with torch.no_grad():
        for batch in loader:
            labels = batch.pop("labels").to(DEVICE)
            batch = {key: value.to(DEVICE) for key, value in batch.items()}
            logits = model(**batch)
            validate_finite_tensor(logits, "Non-finite logits during evaluation.")
            loss = loss_fn(logits, labels)
            validate_finite_tensor(loss, "Non-finite loss during evaluation.")
            preds = logits.argmax(dim=1)
            losses.append(float(loss.item()))
            all_labels.extend(labels.cpu().tolist())
            all_preds.extend(preds.cpu().tolist())
    metrics = compute_metrics(all_labels, all_preds)
    metrics["loss"] = float(np.mean(losses)) if losses else 0.0
    return metrics, all_labels, all_preds


def log_sample_predictions(writer, model, tokenizer, samples, epoch: int, model_name: str):
    model.eval()
    for idx, sample in enumerate(samples):
        encoded = tokenizer(
            sample["text"],
            truncation=True,
            max_length=cfg.max_length,
            return_tensors="pt",
        )
        encoded = {key: value.to(DEVICE) for key, value in encoded.items()}
        with torch.no_grad():
            logits = model(**encoded)
            validate_finite_tensor(
                logits,
                f"Non-finite logits while logging predictions for {model_name}.",
            )
            probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
            pred = int(np.argmax(probs))
        writer.add_text(
            f"samples/example_{idx}",
            (
                f"model={model_name}\\n"
                f"true={LABEL_NAMES[int(sample['label'])]}\\n"
                f"pred={LABEL_NAMES[pred]}\\n"
                f"p_human={probs[0]:.4f}, p_machine={probs[1]:.4f}\\n\\n"
                f"{sample['text'][:500]}"
            ),
            epoch,
        )


fixed_samples = (
    dev_eval_df.groupby("label", group_keys=False)
    .head(2)[["text", "label"]]
    .to_dict(orient="records")
)


In [5]:
def run_experiment(
    experiment_name: str,
    model_name: str,
    train_frame: pd.DataFrame,
    eval_frame: pd.DataFrame,
    training_style: str,
):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    train_loader = make_loader(train_frame, tokenizer, batch_size=cfg.batch_size, shuffle=True)
    eval_loader = make_loader(eval_frame, tokenizer, batch_size=cfg.batch_size, shuffle=False)

    model = EncoderClassifier(model_name=model_name, dropout=cfg.dropout).to(DEVICE)
    train_loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1) if training_style == "article" else nn.CrossEntropyLoss()
    eval_loss_fn = nn.CrossEntropyLoss()

    writer = SummaryWriter(f"{cfg.log_root}/{experiment_name}")
    checkpoint_dir = Path(cfg.checkpoint_root) / experiment_name
    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    best_f1 = -1.0
    best_state = None
    history = []
    current_phase = None
    optimizer = None
    scheduler = None

    for epoch in range(cfg.num_epochs):
        phase = resolve_phase(model_name, epoch)
        set_encoder_trainability(model, model_name=model_name, phase=phase)

        if phase != current_phase:
            phase_steps = max(1, len(train_loader) * phase_block_length(model_name, epoch))
            optimizer = build_optimizer(model, model_name=model_name, phase=phase)
            scheduler = build_scheduler(optimizer, total_steps=phase_steps)
            current_phase = phase

        model.train()
        epoch_losses = []
        progress = tqdm(train_loader, leave=False, desc=f"{experiment_name} epoch {epoch + 1}")

        for step, batch in enumerate(progress, start=1):
            labels = batch.pop("labels").to(DEVICE)
            batch = {key: value.to(DEVICE) for key, value in batch.items()}

            optimizer.zero_grad(set_to_none=True)
            logits = model(**batch)
            validate_finite_tensor(
                logits,
                f"Non-finite logits in {experiment_name}, epoch {epoch + 1}, step {step}.",
            )
            loss = train_loss_fn(logits, labels)
            validate_finite_tensor(
                loss,
                f"Non-finite loss in {experiment_name}, epoch {epoch + 1}, step {step}.",
            )

            loss.backward()
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=cfg.grad_clip_norm)
            validate_finite_tensor(
                grad_norm,
                f"Non-finite gradient norm in {experiment_name}, epoch {epoch + 1}, step {step}.",
            )
            optimizer.step()
            scheduler.step()

            batch_loss = float(loss.item())
            epoch_losses.append(batch_loss)
            progress.set_postfix(
                loss=batch_loss,
                phase=phase,
                lr=f"{optimizer.param_groups[0]['lr']:.2e}",
            )

        train_loss = float(np.mean(epoch_losses)) if epoch_losses else 0.0
        eval_metrics, final_labels, final_preds = evaluate_model(model, eval_loader, eval_loss_fn)

        row = {
            "experiment": experiment_name,
            "epoch": epoch + 1,
            "phase": phase,
            "train_loss": train_loss,
            "eval_loss": eval_metrics["loss"],
            "eval_accuracy": eval_metrics["accuracy"],
            "eval_f1": eval_metrics["f1"],
            "classifier_lr": float(optimizer.param_groups[0]["lr"]),
            "encoder_lr": float(resolve_learning_rates(model_name, phase)[1]) if phase == "full" else 0.0,
        }
        history.append(row)
        print(row)

        writer.add_scalar("loss/train", train_loss, epoch + 1)
        writer.add_scalar("loss/eval", eval_metrics["loss"], epoch + 1)
        writer.add_scalar("metrics/accuracy", eval_metrics["accuracy"], epoch + 1)
        writer.add_scalar("metrics/f1", eval_metrics["f1"], epoch + 1)
        writer.add_scalar("lr/classifier", optimizer.param_groups[0]["lr"], epoch + 1)
        writer.add_scalar("lr/encoder", row["encoder_lr"], epoch + 1)
        writer.add_text("phase", phase, epoch + 1)
        log_sample_predictions(writer, model, tokenizer, fixed_samples, epoch + 1, experiment_name)

        if eval_metrics["f1"] > best_f1:
            best_f1 = eval_metrics["f1"]
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, checkpoint_dir / "best_model.pt")

    if best_state is not None:
        model.load_state_dict(best_state)

    final_metrics, final_labels, final_preds = evaluate_model(model, eval_loader, eval_loss_fn)
    report = classification_report(
        final_labels,
        final_preds,
        target_names=["human", "machine"],
        output_dict=True,
        zero_division=0,
    )
    writer.add_hparams(
        {
            "model_name": model_name,
            "training_style": training_style,
            "train_size": len(train_frame),
            "max_length": cfg.max_length,
            "batch_size": cfg.batch_size,
        },
        {
            "hparam/accuracy": float(final_metrics["accuracy"]),
            "hparam/f1": float(final_metrics["f1"]),
        },
    )
    writer.close()
    return model, pd.DataFrame(history), final_metrics, report


In [6]:
experiments = [
    {
        "experiment_name": "xlm_roberta_original_default",
        "model_name": "xlm-roberta-base",
        "train_frame": train_original_df,
        "training_style": "default",
    },
    {
        "experiment_name": "xlm_roberta_processed_article",
        "model_name": "xlm-roberta-base",
        "train_frame": train_processed_df,
        "training_style": "article",
    },
    {
        "experiment_name": "mdeberta_original_default",
        "model_name": "microsoft/mdeberta-v3-base",
        "train_frame": train_original_df,
        "training_style": "default",
    },
    {
        "experiment_name": "mdeberta_processed_article",
        "model_name": "microsoft/mdeberta-v3-base",
        "train_frame": train_processed_df,
        "training_style": "article",
    },
    {
        "experiment_name": "minilm_original_default",
        "model_name": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
        "train_frame": train_original_df,
        "training_style": "default",
    },
    {
        "experiment_name": "minilm_processed_article",
        "model_name": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
        "train_frame": train_processed_df,
        "training_style": "article",
    },
]

all_histories = []
summary_rows = []
trained_models = {}

for experiment in experiments:
    model, history_df, final_metrics, report = run_experiment(
        experiment_name=experiment["experiment_name"],
        model_name=experiment["model_name"],
        train_frame=experiment["train_frame"],
        eval_frame=dev_eval_df,
        training_style=experiment["training_style"],
    )
    trained_models[experiment["experiment_name"]] = model
    all_histories.append(history_df)
    summary_rows.append(
        {
            "experiment": experiment["experiment_name"],
            "model_name": experiment["model_name"],
            "training_style": experiment["training_style"],
            "train_size": len(experiment["train_frame"]),
            "eval_accuracy": final_metrics["accuracy"],
            "eval_f1": final_metrics["f1"],
            "machine_precision": report["machine"]["precision"],
            "machine_recall": report["machine"]["recall"],
        }
    )

history_table = pd.concat(all_histories, ignore_index=True)
summary_table = pd.DataFrame(summary_rows).sort_values("eval_f1", ascending=False)
display(history_table.tail(12))
display(summary_table)

best_experiment_name = summary_table.iloc[0]["experiment"]
best_model = trained_models[best_experiment_name]
best_experiment_name


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


xlm_roberta_original_default epoch 1:   0%|          | 0/750 [00:00<?, ?it/s]

{'experiment': 'xlm_roberta_original_default', 'epoch': 1, 'phase': 'head', 'train_loss': 0.6889686810175578, 'eval_loss': 0.6916043764733254, 'eval_accuracy': 0.6106666666666667, 'eval_f1': 0.6878674505611972, 'classifier_lr': 0.0, 'encoder_lr': 0.0}


xlm_roberta_original_default epoch 2:   0%|          | 0/750 [00:00<?, ?it/s]

...

minilm_processed_article epoch 5:   0%|          | 0/12768 [00:00<?, ?it/s]

{'experiment': 'minilm_processed_article', 'epoch': 5, 'phase': 'head', 'train_loss': 0.33371675218966673, 'eval_loss': 1.2575177043042285, 'eval_accuracy': 0.5193333333333333, 'eval_f1': 0.3223684210526316, 'classifier_lr': 0.0, 'encoder_lr': 0.0}


,experiment,epoch,phase,train_loss,eval_loss,eval_accuracy,eval_f1,classifier_lr,encoder_lr
18,mdeberta_processed_article,4,head,0.524456,0.503343,0.753000,0.768075,0.000004,0.00000
19,mdeberta_processed_article,5,head,0.522361,0.495773,0.762667,0.775677,0.000000,0.00000
20,minilm_original_default,1,head,0.687547,0.692723,0.478000,0.182672,0.000000,0.00000
21,minilm_original_default,2,full,0.523856,0.696901,0.628333,0.685295,0.000015,0.00001
22,minilm_original_default,3,full,0.337107,0.831940,0.631667,0.675287,0.000007,0.00001
23,minilm_original_default,4,full,0.253933,0.900200,0.637000,0.684073,0.000000,0.00001
24,minilm_original_default,5,head,0.222772,0.911774,0.637667,0.681512,0.000000,0.00000
25,minilm_processed_article,1,head,0.613364,0.806879,0.489667,0.096755,0.000000,0.00000
26,minilm_processed_article,2,full,0.490287,1.119853,0.527667,0.252243,0.000015,0.00001
27,minilm_processed_article,3,full,0.398293,1.162589,0.532000,0.337736,0.000007,0.00001


,experiment,model_name,training_style,train_size,eval_accuracy,eval_f1,machine_precision,machine_recall
3,mdeberta_processed_article,microsoft/mdeberta-v3-base,article,204274,0.762667,0.775677,0.735364,0.820667
1,xlm_roberta_processed_article,xlm-roberta-base,article,204274,0.698667,0.735518,0.655370,0.838000
0,xlm_roberta_original_default,xlm-roberta-base,default,12000,0.629000,0.703753,0.585733,0.881333
4,minilm_original_default,sentence-transformers/paraphrase-multilingual-...,default,12000,0.628333,0.685295,0.594224,0.809333
2,mdeberta_original_default,microsoft/mdeberta-v3-base,default,12000,0.607333,0.674945,0.575800,0.815333
5,minilm_processed_article,sentence-transformers/paraphrase-multilingual-...,article,204274,0.532000,0.337736,0.577419,0.238667


'mdeberta_processed_article'

```bash
tensorboard --logdir runs/task7
```